In [ ]:
import os
import textwrap

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle
import anndata as ad
import gseapy as gp
import seaborn as sns
import scanpy as sc
from scipy.sparse import issparse

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme(style="white")

%matplotlib inline

%load_ext autoreload
%autoreload 2

from eykthyr.eykthyr import Eykthyr, load_anndata

os.makedirs('figure_panels', exist_ok=True)


In [ ]:
e2 = load_anndata('publication_figure_data/human_development.h5ad')

# Panel A

In [ ]:
from popari import tl, pl

tl.preprocess_embeddings(e2.popari, level=0, normalized_key="normalized_X")
tl.compute_empirical_correlations(e2.popari, level=0, feature="X")

tl.leiden(e2.popari, level=0, joint=True, use_rep="normalized_X")


fig, axes = plt.subplots(1, 3, constrained_layout=True, dpi=300, squeeze=False)
_ = pl.in_situ(e2.popari, color="leiden", level=0, 
               size=10, legend_fontsize="large", joint=True, axes=axes)
fig.savefig("figure_panels/Panel_5A.pdf", bbox_inches="tight")


In [ ]:
for ad in e2.popari.datasets:
    ad.obs["cortical_layer"] = "Background"

    ad.obs.loc[ad.obs["leiden"].isin(["5"]), "cortical_layer"] = "Upper cortical (II-III)"
    ad.obs.loc[ad.obs["leiden"].isin(["11", "12"]), "cortical_layer"] = "Middle cortical (IV-V)"
    ad.obs.loc[ad.obs["leiden"].isin(["4"]), "cortical_layer"] = "Deep cortical (VI and subplate)"


layers_to_plot = [
    "Upper cortical (II-III)",
    "Middle cortical (IV-V)",
    "Deep cortical (VI and subplate)"
]


palette = {
    "Upper cortical (II-III)": "#DCEAF7",   # very light blue
    "Middle cortical (IV-V)": "#6BAED6",   # medium blue
    "Deep cortical (VI and subplate)": "#08306B",  # deep navy
    "Background": "#D9D9D9"
}

In [ ]:
fig = sc.pl.spatial(
        e2.popari.datasets[0],
        color="cortical_layer",
        spot_size=30,
        palette=palette,
        groups=layers_to_plot,
        frameon=False,
        show=True,
        return_fig=True        
)
fig.savefig("figure_panels/Panel_5A_0.pdf", bbox_inches="tight")


In [ ]:
fig = sc.pl.spatial(
        e2.popari.datasets[1],
        color="cortical_layer",
        spot_size=45,
        palette=palette,
        groups=layers_to_plot,
        frameon=False,
        show=True,
        return_fig=True        
)
fig.savefig("figure_panels/Panel_5A_1.pdf", bbox_inches="tight")


In [ ]:
fig = sc.pl.spatial(
        e2.popari.datasets[2],
        color="cortical_layer",
        spot_size=45,
        palette=palette,
        groups=layers_to_plot,
        frameon=False,
        show=True,
        return_fig=True        
)
fig.savefig("figure_panels/Panel_5A_2.pdf", bbox_inches="tight")


# Panel B

In [ ]:
highlighted_tfs = ["KLF4", "KLF5", "SP8", "SP9", "MECP2", "YY1", "RXRB"]

timepoint_labels = {
    0: "Second trimester",
    1: "Third trimester",
    2: "Infancy",
}

n_bins = 20
rolling_window = 3
flip_depth = False


def to_dense(x):
    return x.toarray() if issparse(x) else np.asarray(x)


def normalize_vector(x):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmin(x)
    xmax = np.nanmax(x)
    return x / xmax if xmax > 0 else x


def make_depth_profile(depth, value, n_bins=20, rolling_window=3):
    value = normalize_vector(value)

    df = (
        pd.DataFrame({"depth": depth, "value": value})
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    df["depth_bin"] = pd.cut(
        df["depth"],
        bins=n_bins,
        labels=False,
        include_lowest=True
    )

    prof = (
        df.groupby("depth_bin", observed=False)
        .agg(
            depth_mean=("depth", "mean"),
            value_mean=("value", "mean"),
        )
        .dropna()
        .reset_index()
    )

    prof["value_mean"] = prof["value_mean"].rolling(
        window=rolling_window,
        center=True,
        min_periods=1
    ).mean()

    return prof


profile_rows = []

for di, (popari_ad, pert_ad) in enumerate(zip(e2.popari.datasets, e2.perturbed_X)):
    coords = np.asarray(popari_ad.obsm["spatial"])
    depth = coords[:, 1].astype(float)
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-12)

    if flip_depth:
        depth = 1.0 - depth

    X0 = to_dense(pert_ad.obsm["X"])

    for tf in highlighted_tfs:
        key = f"X_{tf}_dropout"
        if key not in pert_ad.obsm:
            print(f"Skipping {tf} in dataset {di}: dropout embedding not found")
            continue

        delta = to_dense(pert_ad.obsm[key]) - X0
        delta_value = np.linalg.norm(delta, axis=1)

        prof = make_depth_profile(
            depth=depth,
            value=delta_value,
            n_bins=n_bins,
            rolling_window=rolling_window,
        )

        prof["dataset_idx"] = di
        prof["timepoint"] = timepoint_labels[di]
        prof["TF"] = tf

        profile_rows.append(prof)

profile_df = pd.concat(profile_rows, ignore_index=True)

global_summary_df = (
    profile_df
    .groupby(["dataset_idx", "timepoint", "depth_bin"], observed=False)
    .agg(
        depth_mean=("depth_mean", "mean"),
        tf_mean=("value_mean", "mean"),
        tf_sem=("value_mean", lambda x: x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
        n_tfs=("value_mean", "size"),
    )
    .reset_index()
)

In [ ]:
plt.figure(figsize=(5.2, 4.2))

for di in sorted(global_summary_df["dataset_idx"].unique()):
    sub = (
        global_summary_df[global_summary_df["dataset_idx"] == di]
        .sort_values("depth_mean")
    )

    plt.plot(
        sub["depth_mean"],
        sub["tf_mean"],
        linewidth=2.5,
        label=timepoint_labels[di]
    )

    plt.fill_between(
        sub["depth_mean"],
        sub["tf_mean"] - sub["tf_sem"],
        sub["tf_mean"] + sub["tf_sem"],
        alpha=0.18
    )

plt.xlabel("Normalized cortical depth")
plt.ylabel("Mean normalized Δ magnitude across TFs")

plt.xticks(
    [0.17, 0.50, 0.83],
    ["Upper\n(II–III)", "Middle\n(IV–V)", "Deep\n(VI+SP)"]
)

plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("figure_panels/Panel_5B.pdf", bbox_inches="tight")
plt.show()

# Panel C

In [ ]:
layers = [
    "Upper cortical (II-III)",
    "Middle cortical (IV-V)",
    "Deep cortical (VI and subplate)",
]

timepoint_labels = {
    0: "Second trimester",
    1: "Third trimester",
    2: "Infancy",
}

stage_colors = {
    "Early neurogenesis\n(Second trimester)": "#4C72B0",   # blue
    "Maturation and myelination\n(Third trimester)": "#DD8452",  # orange
    "Synaptic plasticity\n(Infancy)": "#55A868",           # green
}

tf_to_group = {
    "KLF5": "Early neurogenesis\n(Second trimester)",
    "RXRA": "Maturation and myelination\n(Third trimester)",
    "FEV": "Synaptic plasticity\n(Infancy)",
    "YY1": "Maturation and myelination\n(Third trimester)",
    "ETV3": "Maturation and myelination\n(Third trimester)",
    "RXRB": "Maturation and myelination\n(Third trimester)",
    "TEAD3": "Early neurogenesis\n(Second trimester)",
    "ARID3A": "Early neurogenesis\n(Second trimester)",
    "MECP2": "Synaptic plasticity\n(Infancy)",
    "KLF10": "Synaptic plasticity\n(Infancy)",
    "KLF13": "Synaptic plasticity\n(Infancy)",
    "ZNF354C": "Synaptic plasticity\n(Infancy)",
}

legend_order = [
    "Early neurogenesis\n(Second trimester)",
    "Maturation and myelination\n(Third trimester)",
    "Synaptic plasticity\n(Infancy)",
]

In [ ]:
# Copy cortical layer labels to perturbed objects

for popari_ad, pert_ad in zip(e2.popari.datasets, e2.perturbed_X):
    pert_ad.obs["cortical_layer"] = popari_ad.obs["cortical_layer"].values

# Get all TFs from dropout embeddings

all_tfs = sorted({
    "_".join(k.split("_")[1:-1])
    for k in e2.perturbed_X[0].obsm.keys()
    if k.startswith("X_") and k.endswith("_dropout")
})


# Compute layer-specific TF influence

rows = []

for di, d in enumerate(e2.perturbed_X):
    X0 = d.obsm["X"]

    for tf in all_tfs:
        key = f"X_{tf}_dropout"
        if key not in d.obsm:
            continue

        delta = d.obsm[key] - X0

        row = {
            "dataset_idx": di,
            "timepoint": timepoint_labels[di],
            "TF": tf,
        }

        for layer in layers:
            mask = d.obs["cortical_layer"].values == layer

            row[layer] = (
                float(np.mean(np.abs(delta[mask, :])))
                if mask.sum() > 0
                else np.nan
            )

        rows.append(row)

layer_tf_df = pd.DataFrame(rows)


# Compute developmental rewiring score

rewiring_rows = []

for tf, sub in layer_tf_df.groupby("TF"):
    sub = sub.sort_values("dataset_idx")

    # Require all three developmental stages
    if len(sub) != 3:
        continue

    v0, v1, v2 = sub[layers].astype(float).values

    d01 = np.linalg.norm(v0 - v1)
    d12 = np.linalg.norm(v1 - v2)
    d02 = np.linalg.norm(v0 - v2)

    rewiring_rows.append({
        "TF": tf,
        "T0_to_T1_shift": d01,
        "T1_to_T2_shift": d12,
        "T0_to_T2_shift": d02,
        "rewiring_score": d01 + d12,
        "max_pairwise_shift": max(d01, d12, d02),
    })

rewiring_df = pd.DataFrame(rewiring_rows)

plot_df = (
    rewiring_df
    .sort_values("rewiring_score", ascending=False)
    .head(12)
    .iloc[::-1]
    .copy()
)

plot_df["developmental_group"] = plot_df["TF"].map(tf_to_group)


# Panel D

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))

# stems
ax.vlines(
    x=plot_df["TF"],
    ymin=0,
    ymax=plot_df["rewiring_score"],
    color="black",
    linewidth=2
)

# filled circles with black outlines
for group in legend_order:
    sub = plot_df[plot_df["developmental_group"] == group]

    if sub.empty:
        continue

    ax.scatter(
        sub["TF"],
        sub["rewiring_score"],
        s=55,
        facecolors=stage_colors[group],
        edgecolors="black",
        linewidth=1.2,
        label=group,
        zorder=3
    )

ax.set_ylabel("Developmental rewiring score")
ax.set_title("TFs with largest changes in laminar influence across development")

# clean axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.xticks(rotation=90)

ax.legend(
    frameon=False,
    loc="center left",
    bbox_to_anchor=(0.67, 0.55),
    handlelength=0.8,
    handletextpad=0.4,
    borderaxespad=0
)

plt.tight_layout()
fig.savefig("figure_panels/Panel_5C.pdf", bbox_inches="tight")
plt.show()

In [ ]:
plot_df = heatmap_df.copy()

vmin = plot_df.min().min()
vmax = plot_df.max().max()

norm = Normalize(vmin=vmin, vmax=vmax)

n_rows, n_cols = plot_df.shape
x_labels = plot_df. .tolist()
y_labels = plot_df.index.tolist()

fig, ax = plt.subplots(figsize=(7.5, 4.8))

for i, tf in enumerate(y_labels):
    for j, col in enumerate(x_labels):
        val = plot_df.loc[tf, col]

        # normalize value for plotting
        val_norm = norm(val)

        ax.scatter(
            j, i,
            s=40 + 220 * val_norm,   # size based on normalized value
            c=[[val]],
            cmap="Greens",
            norm=norm,
            edgecolors="black",
            linewidths=0.4
        )

ax.set_xticks(range(n_cols))
ax.set_xticklabels(
    ["U","M","D","U","M","D","U","M","D"],
    fontsize=10
)

ax.set_yticks(range(n_rows))
ax.set_yticklabels(y_labels, fontsize=10)

for x, label in zip([1, 4, 7], ["2nd tri", "3rd tri", "Infancy"]):
    ax.text(
        x, -1.1,
        label,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

for x in [2.5, 5.5]:
    ax.axvline(x, color="lightgray", linewidth=1)

ax.set_xlim(-0.6, n_cols - 0.4)
ax.set_ylim(n_rows - 0.5, -1.4)

ax.set_title(
    "Top rewired TFs show distinct laminar profiles across development",
    fontsize=12,
    pad=12
)

for spine in ["top", "right", "left", "bottom"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="both", length=0)

sm = ScalarMappable(norm=norm, cmap="Greens")
sm.set_array([])

cbar = plt.colorbar(
    sm,
    ax=ax,
    fraction=0.03,
    pad=0.02
)
cbar.set_label("Mean TF perturbation effect (Δ)")

plt.tight_layout()
fig.savefig("figure_panels/Panel_5D.pdf", bbox_inches="tight")
plt.show()

# Panel E

In [ ]:
plt.rcParams["svg.fonttype"] = "none"

mg_idx = 10  # metagene column in obsm["X"]

i = 0
key = f"X_{mg_idx}"
adata = e2.popari.datasets[i]

adata.obs[key] = adata.obsm["X"][:, mg_idx]

fig = sc.pl.spatial(
    adata,
    color=key,
    spot_size=30,
    frameon=False,
    title=f"Dataset {i} — Metagene {mg_idx} expression",
    cmap="viridis",
    colorbar_loc="right",
    show=True,
    return_fig=True,
)
fig.savefig("figure_panels/Panel_5E_0.pdf", bbox_inches="tight")


In [ ]:
i = 1
key = f"X_{mg_idx}"
adata = e2.popari.datasets[i]

adata.obs[key] = adata.obsm["X"][:, mg_idx]

fig = sc.pl.spatial(
    adata,
    color=key,
    spot_size=45,
    frameon=False,
    title=f"Dataset {i} — Metagene {mg_idx} expression",
    cmap="viridis",
    colorbar_loc="right",
    show=True,
    return_fig=True,
)
fig.savefig("figure_panels/Panel_5E_1.pdf", bbox_inches="tight")


In [ ]:
i = 2
key = f"X_{mg_idx}"
adata = e2.popari.datasets[i]

adata.obs[key] = adata.obsm["X"][:, mg_idx]

fig = sc.pl.spatial(
    adata,
    color=key,
    spot_size=45,
    frameon=False,
    title=f"Dataset {i} — Metagene {mg_idx} expression",
    cmap="viridis",
    colorbar_loc="right",
    show=True,
    return_fig=True,
)
fig.savefig("figure_panels/Panel_5E_2.pdf", bbox_inches="tight")


# Panel F

In [ ]:
def plot_metagene_gsea_dotstem(
    enr_results,
    top_n=8,
    term_col="Term",
    pval_col="Adjusted P-value",
    size_col="Odds Ratio",
    title="GSEA Enrichment",
    wrap_width=55,
    figsize=(7, 5.5),
    line_color="#7570b3",
    dot_color="#7570b3",
    min_dot=70,
    max_dot=260,
):
    df = enr_results.copy()
    df = df.sort_values(pval_col, ascending=True).head(top_n).copy()

    df["plot_x"] = -np.log10(df[pval_col].astype(float).clip(lower=1e-300))

    raw_sizes = df[size_col].astype(float)
    smin, smax = raw_sizes.min(), raw_sizes.max()
    if np.isclose(smin, smax):
        df["plot_size"] = (min_dot + max_dot) / 2
    else:
        df["plot_size"] = min_dot + (raw_sizes - smin) / (smax - smin) * (max_dot - min_dot)

    df["plot_label"] = [
        textwrap.fill(str(x), width=wrap_width)
        for x in df[term_col]
    ]

    df = df.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(df))

    fig, ax = plt.subplots(figsize=figsize)

    ax.hlines(y, 0, df["plot_x"], linewidth=2, color=line_color)
    ax.scatter(
        df["plot_x"],
        y,
        s=df["plot_size"],
        color=dot_color,
        edgecolor="none",
        zorder=3,
    )

    ax.set_yticks(y)
    ax.set_yticklabels(df["plot_label"], fontsize=12)
    ax.set_xlabel(r"$-\log_{10}(\mathrm{Adjusted\ P\text{-}value})$", fontsize=14)
    ax.set_title(title, fontsize=18, weight="bold", pad=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(left=0)
    ax.grid(False)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%d"))

    fig.tight_layout()
    return fig, ax, df

In [ ]:
from popari.util import get_metagene_signature

def do_metagene_gsea(
    dataset,
    metagene_index,
    metagene_key="M",
    sensitivity=0.5,
    gene_sets="GO_Biological_Process_2023",
    organism="mouse",
):
    metagenes = dataset.uns[metagene_key][dataset.name]
    metagene = metagenes[:, metagene_index]
    gene_names = dataset.var_names

    signature = list(
        get_metagene_signature(
            metagene,
            gene_names,
            show_plot=False,
            sensitivity=sensitivity,
        )
    )

    enr = gp.enrichr(
        gene_list=signature,
        gene_sets=gene_sets,   # single library only
        organism=organism,
        background=list(gene_names),
        outdir=None,
    )

    return enr, signature

In [ ]:
enr_go, signature = do_metagene_gsea(
    dataset=e2.popari.datasets[0],
    metagene_index=10,
    sensitivity=0.5,
    gene_sets="GO_Biological_Process_2023",
)

go_df = enr_go.results.sort_values("Adjusted P-value").copy()


fig_pretty, ax_pretty, plot_df = plot_metagene_gsea_dotstem(
    go_df,
    top_n=5,
    size_col="Odds Ratio",
    title="Metagene 10 expression\nGSEA Enrichment",
    figsize=(7, 5.5),
)

plt.show()

outpath = "figure_panels/Panel_5F.pdf"
fig_pretty.savefig(outpath, format="pdf", bbox_inches="tight")
plt.close(fig_pretty)

# Panel G

In [ ]:
datasets = e2.perturbed_X
mg_idx = [6, 10]

# TF list
allTFs = sorted({
    "_".join(k.split("_")[1:-1])
    for k in datasets[0].obsm.keys()
    if k.startswith("X_") and k.endswith("_dropout")
})

mg_names = [f"MG_{i}" for i in mg_idx]

rows = []
for di, d in enumerate(datasets):
    X0 = d.obsm["X"]

    for TF in allTFs:
        k = f"X_{TF}_dropout"
        if k not in d.obsm:
            continue

        delta = d.obsm[k] - X0                    # (cells, metagenes)
        per_mg = np.mean(np.abs(delta), axis=0)   # (n_metagenes,)

        row = {"dataset_idx": di, "TF": TF}
        for i, name in zip(mg_idx, mg_names):
            row[name] = float(per_mg[i])          # keep which metagene is which

        row["mean_abs_change_mg"] = float(np.mean([row[n] for n in mg_names]))

        rows.append(row)

df = pd.DataFrame(rows)

df_rank = df.sort_values(["dataset_idx", "mean_abs_change_mg"], ascending=[True, False])
df_rank.groupby("dataset_idx").head(20)

for name in mg_names:
    print(f"\nTop TFs for {name}")
    display(df.sort_values(["dataset_idx", name], ascending=[True, False])
              .groupby("dataset_idx").head(10)[["dataset_idx", "TF", name]])


In [ ]:
highlight_tfs = {
    "MG_10": ["KLF4", "KLF5", "SP8", "SP9"]
}

cmap = "RdBu_r"
figsize = (6.2, 4.6)

mats = {}
for mg, tf_list in highlight_tfs.items():
    mat = (
        df[df["TF"].isin(tf_list)]
          .pivot_table(index="TF", columns="dataset_idx", values=mg, aggfunc="mean")
          .reindex(tf_list)
          .fillna(0.0)
    )
    mats[mg] = mat

fig, axes = plt.subplots(len(mats), 1, figsize=figsize, sharex=True)
if len(mats) == 1:
    axes = [axes]

for ax, (mg, mat) in zip(axes, mats.items()):
    vals = mat.to_numpy()

    local_max = float(np.abs(vals).max())
    if local_max == 0:
        local_max = 1.0

    n_rows, n_cols = vals.shape

    # pcolormesh wants "bin edges"
    x = np.arange(n_cols + 1)
    y = np.arange(n_rows + 1)

    pm = ax.pcolormesh(
        x, y, vals,
        cmap=cmap,
        vmin=-local_max,
        vmax=local_max,
        shading="flat",
        edgecolors="0.85",   # grid lines
        linewidth=0.6
    )

    # Put first TF at top (like imshow)
    ax.set_ylim(n_rows, 0)

    # ticks at cell centers
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(mat.index, fontsize=10)

    ax.set_title(mg, fontsize=12)
    ax.tick_params(axis="both", length=0)

    cbar = fig.colorbar(pm, ax=ax, fraction=0.04, pad=0.03)
    cbar.set_label("Effect size")

# X labels (centers)
last_mat = list(mats.values())[-1]
n_cols = last_mat.shape[1]
axes[-1].set_xticks(np.arange(n_cols) + 0.5)
axes[-1].set_xticklabels(["Second trimester", "Third trimester", "Infancy"])
axes[-1].tick_params(axis="x", labelrotation=15)

plt.tight_layout()


fig.savefig("figure_panels/Panel_5G.pdf", bbox_inches="tight")
plt.show()


# Panel H

In [ ]:
plt.rcParams["svg.fonttype"] = "none"

mg_idx = 6  # metagene column in obsm["X"]


i = 0
key = f"X_{mg_idx}"
adata = e2.popari.datasets[i]

adata.obs[key] = adata.obsm["X"][:, mg_idx]

fig = sc.pl.spatial(
    adata,
    color=key,
    spot_size=25,
    frameon=False,
    title=f"Dataset {i} — Metagene {mg_idx} expression",
    cmap="viridis",
    colorbar_loc="right",
    show=True,
    return_fig=True,
)
fig.savefig("figure_panels/Panel_5H_0.pdf", bbox_inches="tight")


In [ ]:
i = 1
key = f"X_{mg_idx}"
adata = e2.popari.datasets[i]

adata.obs[key] = adata.obsm["X"][:, mg_idx]

fig = sc.pl.spatial(
    adata,
    color=key,
    spot_size=35,
    frameon=False,
    title=f"Dataset {i} — Metagene {mg_idx} expression",
    cmap="viridis",
    colorbar_loc="right",
    show=True,
    return_fig=True,
)
fig.savefig("figure_panels/Panel_5H_1.pdf", bbox_inches="tight")


In [ ]:
i = 2
key = f"X_{mg_idx}"
adata = e2.popari.datasets[i]

adata.obs[key] = adata.obsm["X"][:, mg_idx]

fig = sc.pl.spatial(
    adata,
    color=key,
    spot_size=35,
    frameon=False,
    title=f"Dataset {i} — Metagene {mg_idx} expression",
    cmap="viridis",
    colorbar_loc="right",
    show=True,
    return_fig=True,
)
fig.savefig("figure_panels/Panel_5H_2.pdf", bbox_inches="tight")


# Panel I

In [ ]:
enr_go, signature = do_metagene_gsea(
    dataset=e2.popari.datasets[0],
    metagene_index=6,
    sensitivity=0.5,
    gene_sets="GO_Biological_Process_2023",
)

go_df = enr_go.results.sort_values("Adjusted P-value").copy()

fig_pretty, ax_pretty, plot_df = plot_metagene_gsea_dotstem(
    go_df,
    top_n=5,
    size_col="Odds Ratio",
    title="Metagene 6 expression\nGSEA Enrichment",
    figsize=(7, 5.5),
)

plt.show()

outpath = "figure_panels/Panel_5I.pdf"
fig_pretty.savefig(outpath, format="pdf", bbox_inches="tight")
plt.close(fig_pretty)

# Panel J

In [ ]:
highlight_tfs = {
    "MG_6":  ["MECP2", "YY1", "RXRB", "HLTF"]
}

cmap = "RdBu_r"
figsize = (6.2, 4.6)

mats = {}
for mg, tf_list in highlight_tfs.items():
    mat = (
        df[df["TF"].isin(tf_list)]
          .pivot_table(index="TF", columns="dataset_idx", values=mg, aggfunc="mean")
          .reindex(tf_list)
          .fillna(0.0)
    )
    mats[mg] = mat

fig, axes = plt.subplots(len(mats), 1, figsize=figsize, sharex=True)
if len(mats) == 1:
    axes = [axes]

for ax, (mg, mat) in zip(axes, mats.items()):
    vals = mat.to_numpy()

    local_max = float(np.abs(vals).max())
    if local_max == 0:
        local_max = 1.0

    n_rows, n_cols = vals.shape

    # pcolormesh wants "bin edges"
    x = np.arange(n_cols + 1)
    y = np.arange(n_rows + 1)

    pm = ax.pcolormesh(
        x, y, vals,
        cmap=cmap,
        vmin=-local_max,
        vmax=local_max,
        shading="flat",
        edgecolors="0.85",   # grid lines
        linewidth=0.6
    )

    # Put first TF at top (like imshow)
    ax.set_ylim(n_rows, 0)

    # ticks at cell centers
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(mat.index, fontsize=10)

    ax.set_title(mg, fontsize=12)
    ax.tick_params(axis="both", length=0)

    cbar = fig.colorbar(pm, ax=ax, fraction=0.04, pad=0.03)
    cbar.set_label("Effect size")

# X labels (centers)
last_mat = list(mats.values())[-1]
n_cols = last_mat.shape[1]
axes[-1].set_xticks(np.arange(n_cols) + 0.5)
axes[-1].set_xticklabels(["Second trimester", "Third trimester", "Infancy"])
axes[-1].tick_params(axis="x", labelrotation=15)

plt.tight_layout()


fig.savefig("figure_panels/Panel_5J.pdf", bbox_inches="tight")
plt.show()
